# Stage 2 — Multi-label CNN classifier (Colab)

Standalone notebook for training the Stage 2 CNN on the crop dataset produced by Chapter 10 of `01_main_pipeline.ipynb`.

**Why a separate Colab notebook?** Stage 2 is built with Keras / TensorFlow. TensorFlow does not support GPU on native Windows from version 2.11 onwards, so training on the project workstation would fall back to CPU and take many hours. Colab provides a free GPU (T4 or L4) and the training takes minutes.

**Workflow:**
1. Upload `stage2_crops.zip` to `MyDrive/PPE_Project/` on your Google Drive.
2. Open this notebook in Colab and select **Runtime → Change runtime type → GPU**.
3. Run all cells.
4. The trained weights (`best.keras`) and training history (`history.csv`) for both models, plus a comparison plot, will be written to `MyDrive/PPE_Project/stage2_results/`.
5. Sync the results back to the laptop and commit them under `results/stage2/` in the repo.

**What this notebook does (in order):**
1. Mount Google Drive
2. Extract `stage2_crops.zip` into the local `/content/` filesystem
3. Load the four splits into `tf.data` datasets
4. Build a transfer-learning classifier with a *frozen* ImageNet backbone (feature extraction, per Unit 8 of the course)
5. Train **VGG16** for 20 epochs
6. Train **ResNet50** for 20 epochs
7. Plot training curves side-by-side
8. Save all artifacts to Drive


## Before you run

1. **Upload `stage2_crops.zip` to Drive** under `MyDrive/PPE_Project/`. The zip is ~210 MB.
2. **Change runtime to GPU**: `Runtime → Change runtime type → Hardware accelerator: T4 GPU (or L4 GPU)`.
3. **Run all cells** (`Runtime → Run all`).


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT      = "/content/drive/MyDrive/PPE_Project"
ZIP_PATH        = f"{DRIVE_ROOT}/stage2_crops.zip"
RESULTS_DIR     = f"{DRIVE_ROOT}/stage2_results"
EXTRACT_TO      = "/content/stage2_crops"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(ZIP_PATH), f"Missing: {ZIP_PATH}. Upload stage2_crops.zip to {DRIVE_ROOT} first."
print(f"Found zip: {ZIP_PATH}")
print(f"Results will go to: {RESULTS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found zip: /content/drive/MyDrive/PPE_Project/stage2_crops.zip
Results will go to: /content/drive/MyDrive/PPE_Project/stage2_results


In [5]:
# Extract zip (handles PowerShell's backslash-path bug)
import zipfile, os, shutil, time

# Clean any partial extraction
if os.path.exists(EXTRACT_TO):
    shutil.rmtree(EXTRACT_TO)
os.makedirs(EXTRACT_TO)

print(f"Extracting {ZIP_PATH} -> {EXTRACT_TO} (with path normalization)...")
t0 = time.time()
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for info in z.infolist():
        # PowerShell Compress-Archive uses backslash; Linux needs forward slash
        name = info.filename.replace("\\", "/")
        if name.endswith("/"):
            continue  # directories are created on-demand below
        target = os.path.join(EXTRACT_TO, name)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)
print(f"Done in {time.time() - t0:.1f}s")

# Verify
import pandas as pd
for split in ("train", "val", "test_in_domain", "test_out_of_domain"):
    csv = f"{EXTRACT_TO}/{split}/labels.csv"
    crops_dir = f"{EXTRACT_TO}/{split}/crops"
    if os.path.exists(csv):
        n_csv = sum(1 for _ in open(csv)) - 1
        n_files = len(os.listdir(crops_dir))
        print(f"  {split:<22s} csv={n_csv:>6,}  files={n_files:>6,}  match={n_csv == n_files}")
    else:
        print(f"  {split:<22s} MISSING")


Extracting /content/drive/MyDrive/PPE_Project/stage2_crops.zip -> /content/stage2_crops (with path normalization)...
Done in 5.1s
  train                  csv= 8,796  files= 8,796  match=True
  val                    csv=   150  files=   150  match=True
  test_in_domain         csv=   155  files=   155  match=True
  test_out_of_domain     csv= 2,243  files= 2,243  match=True


In [6]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TF version          : {tf.__version__}")
print(f"GPU available       : {bool(tf.config.list_physical_devices('GPU'))}")
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    print(f"  - {g}")


TF version          : 2.20.0
GPU available       : True
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [7]:
# Hyperparameters
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 20
INITIAL_LR   = 1e-4

# Feature-extraction (frozen backbone). Per Unit 8 of the course.
FREEZE_BACKBONE = True


In [8]:
def load_split(split_name, shuffle=False):
    """Load a split into a tf.data.Dataset of (image, [helmet, vest])."""
    base = f"{EXTRACT_TO}/{split_name}"
    df = pd.read_csv(f"{base}/labels.csv")
    paths  = [f"{base}/crops/{fn}" for fn in df["filename"]]
    labels = df[["helmet", "vest"]].values.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(df)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, len(df)


train_ds, n_train = load_split("train", shuffle=True)
val_ds,   n_val   = load_split("val",   shuffle=False)

print(f"Train : {n_train:,} crops  ({n_train // BATCH_SIZE} batches)")
print(f"Val   : {n_val:,} crops  ({(n_val + BATCH_SIZE - 1) // BATCH_SIZE} batches)")


Train : 8,796 crops  (274 batches)
Val   : 150 crops  (5 batches)


In [9]:
# Augmentation block (applied only at training time)
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.05, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
    layers.RandomBrightness(0.2, seed=SEED),
    layers.RandomContrast(0.2, seed=SEED),
], name="augmentation")


def build_classifier(backbone_fn, preprocess_fn, name):
    """Multi-label classifier: frozen ImageNet backbone + small Dense head."""
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = preprocess_fn(x)

    backbone = backbone_fn(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = not FREEZE_BACKBONE
    x = backbone(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(2, activation="sigmoid", name="helmet_vest")(x)

    model = keras.Model(inputs, outputs, name=name)
    return model


def train_one(backbone_fn, preprocess_fn, name):
    """Compile, train, and save weights + history. Returns history dict."""
    print(f"\n{'='*60}\nTraining {name}\n{'='*60}")

    model = build_classifier(backbone_fn, preprocess_fn, name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    out_dir       = f"{RESULTS_DIR}/{name}"
    os.makedirs(out_dir, exist_ok=True)
    weights_path  = f"{out_dir}/best.keras"
    history_path  = f"{out_dir}/history.csv"

    cbs = [
        keras.callbacks.ModelCheckpoint(weights_path, save_best_only=True,
                                        monitor="val_auc", mode="max"),
        keras.callbacks.EarlyStopping(patience=8, monitor="val_auc", mode="max",
                                      restore_best_weights=True),
        keras.callbacks.CSVLogger(history_path),
    ]

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=cbs, verbose=2,
    )

    print(f"Best weights -> {weights_path}")
    print(f"History      -> {history_path}")
    return history.history


In [10]:
vgg_hist = train_one(
    applications.VGG16,
    applications.vgg16.preprocess_input,
    "vgg16",
)



Training vgg16
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/20
275/275 - 60s - 218ms/step - accuracy: 0.5649 - auc: 0.6375 - loss: 0.9434 - precision: 0.4584 - recall: 0.4574 - val_accuracy: 0.5533 - val_auc: 0.7816 - val_loss: 0.6051 - val_precision: 0.6275 - val_recall: 0.5766
Epoch 2/20
275/275 - 43s - 157ms/step - accuracy: 0.6002 - auc: 0.7733 - loss: 0.5607 - precision: 0.6111 - recall: 0.5433 - val_accuracy: 0.6467 - val_auc: 0.8160 - val_loss: 0.5287 - val_precision: 0.6977 - val_recall: 0.5405
Epoch 3/20
275/275 - 47s - 171ms/step - accuracy: 0.6354 - auc: 0.8141 - loss: 0.4909 - precision: 0.6674 - recall: 0.5763 - val_accuracy: 0.6067 - val_auc: 0.8281 - val_loss: 0.5066 - val_precision: 0.7386 - val_recall: 0.5856
Epoch 4/20
275/275 - 47s - 172ms/step - accuracy: 0.6438 - auc: 0.8374 - loss: 0.4575 - precision: 0.7046 - recall: 0.5881 - val_accuracy: 0.6200 - val_auc: 0.8354 - val_loss: 0.4980 - val_precision: 0.7556 - val_recall: 0.6126
Epoch 5/20
275/275 - 

In [ ]:
resnet_hist = train_one(
    applications.ResNet50,
    applications.resnet50.preprocess_input,
    "resnet50",
)



Training resnet50
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/20
275/275 - 62s - 225ms/step - accuracy: 0.5799 - auc: 0.7889 - loss: 0.5065 - precision: 0.6660 - recall: 0.5209 - val_accuracy: 0.6333 - val_auc: 0.8540 - val_loss: 0.4665 - val_precision: 0.7949 - val_recall: 0.5586
Epoch 2/20
275/275 - 43s - 158ms/step - accuracy: 0.6096 - auc: 0.8821 - loss: 0.3925 - precision: 0.7700 - recall: 0.6394 - val_accuracy: 0.6400 - val_auc: 0.8828 - val_loss: 0.4271 - val_precision: 0.8272 - val_recall: 0.6036
Epoch 3/20
275/275 - 45s - 162ms/step - accuracy: 0.6128 - auc: 0.9053 - loss: 0.3558 - precision: 0.8025 - recall: 0.6874 - val_accuracy: 0.7067 - val_auc: 0.9030 - val_loss: 0.3911 - val_precision: 0.8352 - val_recall: 0.6847
Epoch 4/20
275/275 - 43s - 157ms/step - accuracy: 0.6240 - auc: 0.9173 - loss: 0.3344 - precision: 0.8149 - recall: 0.7075 - val_accuracy: 0.6400 - val_auc: 0.9007 - val_loss: 0.3866 - val_precision: 0.8085 - val_recall: 0.6847
Epoch 5/20


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for hist, name, color in [(vgg_hist, "VGG16", "#1e40af"),
                          (resnet_hist, "ResNet50", "#dc2626")]:
    epochs = range(1, len(hist["loss"]) + 1)
    axes[0].plot(epochs, hist["loss"],     color=color, linestyle="-",  label=f"{name} train")
    axes[0].plot(epochs, hist["val_loss"], color=color, linestyle="--", label=f"{name} val")

    axes[1].plot(epochs, hist["auc"],     color=color, linestyle="-",  label=f"{name} train")
    axes[1].plot(epochs, hist["val_auc"], color=color, linestyle="--", label=f"{name} val")

    axes[2].plot(epochs, hist["accuracy"],     color=color, linestyle="-",  label=f"{name} train")
    axes[2].plot(epochs, hist["val_accuracy"], color=color, linestyle="--", label=f"{name} val")

for ax, title, ylabel in zip(axes,
                              ["Loss (BCE)", "AUC", "Accuracy"],
                              ["loss", "AUC", "accuracy"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Stage 2 - VGG16 vs ResNet50 training curves (feature extraction, frozen backbone)",
             fontsize=12, y=1.02)
plt.tight_layout()
plot_path = f"{RESULTS_DIR}/comparison.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nSaved plot to: {plot_path}")


In [ ]:
# Final-epoch numbers (best-restored weights, since restore_best_weights=True)
print("=" * 70)
print("Best (restored) validation metrics")
print("=" * 70)
print(f"{'Model':<12s} {'loss':>8s} {'acc':>8s} {'AUC':>8s} {'P':>8s} {'R':>8s}")
print("-" * 70)
for hist, name in [(vgg_hist, "VGG16"), (resnet_hist, "ResNet50")]:
    # Use the best-AUC index (matches the saved best.keras)
    best_idx = int(np.argmax(hist["val_auc"]))
    print(
        f"{name:<12s} "
        f"{hist['val_loss'][best_idx]:>8.4f} "
        f"{hist['val_accuracy'][best_idx]:>8.4f} "
        f"{hist['val_auc'][best_idx]:>8.4f} "
        f"{hist['val_precision'][best_idx]:>8.4f} "
        f"{hist['val_recall'][best_idx]:>8.4f}"
    )
print()
print(f"All artifacts under: {RESULTS_DIR}")
print(f"  vgg16/best.keras       <- trained weights for VGG16")
print(f"  vgg16/history.csv      <- per-epoch metrics")
print(f"  resnet50/best.keras    <- trained weights for ResNet50")
print(f"  resnet50/history.csv   <- per-epoch metrics")
print(f"  comparison.png         <- side-by-side training curves")


## Next steps

1. The trained models are now in `MyDrive/PPE_Project/stage2_results/`.
2. Sync them back to the laptop (Drive client or manual download).
3. Place them under `results/stage2/` in the repo on the laptop.
4. Commit and push.
5. On the workstation, `git pull` to pick up the weights.
6. Continue in `01_main_pipeline.ipynb` with Chapter 12 (Stage 2 evaluation on the two test sets).
